# Photonic Entanglement Characterization Workflow

This notebook demonstrates an end-to-end two-qubit photonic analysis workflow with the current `pec` package. It uses representative Bell-state tomography counts from the repository notebooks and shows how to structure lab data, reconstruct a density matrix, evaluate Bell-state overlap, and report CHSH quantities in a clean, experiment-facing format.

## Workflow

The analysis below follows the same broad sequence used in the original notebooks:

1. import the package from `src/`,
2. standardize a lab-style count table,
3. reconstruct a physical two-qubit state,
4. compute purity and Bell-state fidelities,
5. compute CHSH correlators and the resulting `S` value,
6. present the reconstructed state and derived quantities with compact plots.

In [ ]:
from pathlib import Path
import sys

search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_roots if (path / "src" / "pec").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root containing src/pec.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Repository root: {repo_root}")
print(f"Using package source: {src_path}")

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from pec import bell
from pec import chsh
from pec import io
from pec import metrics
from pec import plotting
from pec import states
from pec import tomography

np.set_printoptions(precision=3, suppress=True)
pd.options.display.float_format = "{:.4f}".format

## 1. Structure Bell-tomography count data

This example uses the 16-setting two-qubit coincidence table from `Bell_PreLab-Final.ipynb`. In routine use, the same workflow can start from `pec.io.load_counts_table(...)` on a CSV, TSV, or spreadsheet export. To keep this notebook self-contained, the table is created in memory in a spreadsheet-like form and then standardized with `pec.io`.

In [ ]:
raw_tomography_export = pd.DataFrame(
    {
        "Measurement State": [
            "HH", "HV", "VH", "VV",
            "HD", "HL", "DH", "RH",
            "DD", "RD", "RL", "DR",
            "DV", "RV", "VD", "VL",
        ],
        "Coincidence Counts": [
            34749, 324, 444, 35805,
            17238, 16722, 16901, 16324,
            32028, 15132, 33586, 17932,
            13441, 17521, 13171, 17170,
        ],
    }
)

# For a real spreadsheet export, the same standardized table can be loaded with:
# counts_table = io.load_counts_table(repo_root / "path" / "bell_tomography_counts.csv")

counts_table = io.standardize_counts_table(raw_tomography_export)
tomography_counts = io.counts_dict_from_table(raw_tomography_export)

counts_table

In [ ]:
plotting.plot_coincidence_counts(
    counts_table["counts"],
    title="Two-qubit tomography coincidence counts",
    ylabel="Coincidence counts",
)

## 2. Assemble measurement operators and reconstruct the state

The tomography helpers convert polarization labels such as `HH`, `RL`, or `DV` into projectors automatically. For raw coincidence counts, the physically constrained maximum-likelihood path is the appropriate high-level entry point.

In [ ]:
measurement_labels = list(counts_table.index)
measurement_operators = tomography.measurement_projectors_from_labels(measurement_labels)

rho_reconstructed = tomography.reconstruct_density_matrix(
    tomography_counts,
    measurement_operators=measurement_operators,
    measurement_labels=measurement_labels,
    method="mle",
)

pd.DataFrame(
    rho_reconstructed,
    index=["HH", "HV", "VH", "VV"],
    columns=["HH", "HV", "VH", "VV"],
)

## 3. Summarize the reconstructed state

A useful first summary is trace, purity, and fidelity to a target Bell state. The same summary can also include the CHSH value for a standard set of analyzer directions in the `x-z` plane.

In [ ]:
z_axis = bell.unit_vector(0.0, 0.0)
x_axis = bell.unit_vector(np.pi / 2.0, 0.0)
b_axis = (z_axis + x_axis) / np.sqrt(2.0)
bp_axis = (z_axis - x_axis) / np.sqrt(2.0)

phi_plus = states.bell_state("phi_plus")
purity_value = metrics.purity(rho_reconstructed)
phi_plus_fidelity = metrics.fidelity_pure(rho_reconstructed, phi_plus)

summary = tomography.reconstruction_summary(
    rho_reconstructed,
    target_state=phi_plus,
    chsh_settings=(z_axis, x_axis, b_axis, bp_axis),
)

summary_table = pd.DataFrame(
    {
        "Quantity": [
            "Trace",
            "Purity",
            "Fidelity to Phi+",
            "Dominant Bell state",
            "CHSH S",
        ],
        "Value": [
            summary["trace"],
            purity_value,
            phi_plus_fidelity,
            summary["dominant_bell_state"],
            summary["chsh_s"],
        ],
    }
)

summary_table

In [ ]:
plotting.plot_density_matrix(
    rho_reconstructed,
    title="Reconstructed two-qubit density matrix",
    annotate=True,
    colorbar=True,
)

## 4. Bell-state analysis

Bell-state fidelities quantify how closely the reconstructed state aligns with the canonical entangled basis. A state dominated by `Phi+` is consistent with the Bell-pair preparation used in the source notebook.

In [ ]:
bell_fidelities = bell.bell_state_fidelities(rho_reconstructed)

bell_fidelity_table = pd.DataFrame(
    {
        "Bell state": ["Phi+", "Phi-", "Psi+", "Psi-"],
        "Fidelity": [
            bell_fidelities["phi_plus"],
            bell_fidelities["phi_minus"],
            bell_fidelities["psi_plus"],
            bell_fidelities["psi_minus"],
        ],
    }
).set_index("Bell state")

display(bell_fidelity_table)
plotting.plot_bell_state_fidelities(
    bell_fidelities,
    title="Bell-state fidelities for the reconstructed state",
)

## 5. CHSH correlators and `S`

For a Bell-like state, the standard `x-z` plane settings `a = z`, `a' = x`, `b = (z + x) / sqrt(2)`, and `b' = (z - x) / sqrt(2)` provide the textbook CHSH evaluation. The correlators and `S` below are computed directly from the reconstructed density matrix.

In [ ]:
chsh_correlators = chsh.correlators_from_rho(
    rho_reconstructed,
    z_axis,
    x_axis,
    b_axis,
    bp_axis,
)
s_value = chsh.chsh_s_from_rho(
    rho_reconstructed,
    z_axis,
    x_axis,
    b_axis,
    bp_axis,
)

chsh_table = pd.DataFrame(
    {
        "Quantity": ["E(a,b)", "E(a,b')", "E(a',b)", "E(a',b')", "S"],
        "Value": [
            chsh_correlators["ab"],
            chsh_correlators["abp"],
            chsh_correlators["apb"],
            chsh_correlators["apbp"],
            s_value,
        ],
    }
).set_index("Quantity")

display(chsh_table)
plotting.plot_chsh_correlators(
    chsh_correlators,
    title="CHSH correlators from the reconstructed state",
)

## Interpretation

This notebook collects the core photonic characterization outputs in one place: a standardized lab-style count table, a physical two-qubit density matrix, Bell-basis fidelities, and a CHSH estimate. In a typical Bell-pair experiment, a dominant `Phi+` fidelity together with `|S| > 2` indicates entanglement and Bell-inequality violation, while the density-matrix heatmap provides an immediate visual check of the reconstructed coherence structure.